[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nammaschool/interactive-learning/blob/main/notebooks/courses/current_electricity.ipynb)

# Current electricity full course: the practice set, in Python

Twenty-eight exercises in three tiers: **tier 1** warm-ups (one idea, one step),
**tier 2** core problems (the level of the course's worked problems) and **tier 3** stretch problems (run a formula
backwards, combine two ideas, spot the trap). Each tier is harder than the last.

This notebook also has the course's **simulations**: electrons zigzagging through a wire while their average drifts,
a cell's voltage falling as it gives more current, a circuit solver that finds every current by letting the junctions
settle (the same answers as Kirchhoff's rules), the Wheatstone bridge finding its balance, and the e-bike's battery
pack. You can move all five yourself: the **field and the collisions**, the **cell's EMF and internal resistance**,
the **two cells and the resistor** of the network, the **fourth arm** of the bridge, and the **layout and load** of
the pack.

**How to use this notebook.** The grey boxes are cells. Click the first one and press **Shift + Enter** (or the ▶ Run
button) to run it. Run each cell in turn, from the top down. When sliders appear, move them and watch: the picture
redraws by itself. Change any number and run the cell again: nothing can break. It works in your browser (no install)
and in Google Colab.

**Typing answers.** Work the question out first (on paper, or in a cell of your own), then type your answer into the
box and press **check**: it tells you which parts are right. Press **show working** to see the steps. Give the answer
in the unit the question asks for, to two decimals or more. Big and small numbers in E notation: `8.5e28` for
8.5 × 10²⁸, `1.6e-19` for 1.6 × 10⁻¹⁹ (`8.5*10^28` works too). You can type the arithmetic: `10/(8.5e28*1.6e-19*2.5e-6)*1000`,
`1.68e-8*25/1.5e-6`, `1/(1/6+1/3)`. Numbers without thousands separators: `5400`, not `5,400`. For several answers,
separate them with commas, in the order the question lists them (the box's label shows the order). You can also put
your answer between the quotes in a cell's last line, `practice("P1.4", "")`, and run the cell: it checks straight
away.

**The cheat sheet.** Every formula and constant from the course on two pages:
[current electricity cheat sheet (PDF)](https://nammaschool.github.io/interactive-learning/cheatsheets/current_electricity_cheat_sheet.pdf).

In [ ]:
# Setup: in the browser version (JupyterLite) the sliders need one install; elsewhere this does nothing.
import sys
if "pyodide" in sys.modules:
    import piplite
    await piplite.install(["ipywidgets"])

In [ ]:
# The look, the constants, the circuit solver and the checker: run this cell first.
import ast, math, operator, random, re, textwrap
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display
from ipywidgets import interact, FloatSlider, IntSlider, Text, Button, Output, HBox, VBox, Layout

SLATE = "#1f2b28"                                     # the slate ground
CHALK, DIM, GHOST = "#ece6d6", "#93a097", "#3b4c47"   # wires and lines in chalk, labels dim, ghost lines
SKY, MINT, CORAL, VIOLET, GOLD = "#6fb6bb", "#a0c27c", "#e3896b", "#b49ad6", "#e8c56b"
# current SKY · the field CORAL · energy and power VIOLET · wires and lines CHALK · the one mark GOLD

plt.rcParams.update({
    "figure.facecolor": SLATE, "axes.facecolor": SLATE, "savefig.facecolor": SLATE,
    "text.color": CHALK, "axes.labelcolor": CHALK, "axes.edgecolor": DIM, "axes.titlecolor": CHALK,
    "xtick.color": DIM, "ytick.color": DIM, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "axes.axisbelow": True, "grid.color": GHOST, "grid.linewidth": 0.8,
    "font.size": 13, "axes.titlesize": 15, "axes.labelsize": 13, "legend.fontsize": 12,
    "lines.linewidth": 2.4, "figure.dpi": 100,
})

def footnote(fig, text="simulated · checked against the formula"):
    fig.text(0.99, 0.01, text, color=DIM, fontsize=10, ha="right", va="bottom", family="monospace")

# ---- the constants (the course's rounded values; the same in every question) ----
E = 1.60e-19             # C, the size of one electron's charge
ME = 9.11e-31            # kg, an electron's mass
N_CU = 8.5e28            # free electrons in each cubic metre of copper
RHO_CU = 1.68e-8         # Ω m, copper's resistivity at 20 °C
ALPHA_CU = 0.0039        # per °C, copper's temperature coefficient
RHO_NICHROME = 1.0e-6    # Ω m, nichrome's resistivity at 20 °C

# ---- the course's circuit solver: no loop equations, the junctions just settle ----
def settle(nodes, branches, fixed, sweeps=20000, tol=1e-13):
    """nodes: names; branches: (from, to, R, emf), where emf is the rise in potential met going from → to through the
    branch's cell (0 for a plain resistor); fixed: {node: potential}. Each free junction's potential is moved, again and
    again, to the value at which what flows in equals what flows out. Returns ({node: V}, [current from → to in each
    branch])."""
    V = {n: fixed.get(n, 0.0) for n in nodes}
    for _ in range(sweeps):
        worst = 0.0
        for n in nodes:
            if n in fixed: continue
            g = s = 0.0
            for a, b, R, emf in branches:
                if a == n: g += 1 / R; s += (V[b] - emf) / R
                elif b == n: g += 1 / R; s += (V[a] + emf) / R
            new = s / g
            worst = max(worst, abs(new - V[n])); V[n] = new
        if worst < tol: break
    return V, [(V[a] - V[b] + emf) / R for a, b, R, emf in branches]

def close(a, b, rel=1e-4):
    """True when a simulated or settled value matches the formula (to a ten-thousandth by default)."""
    return abs(a - b) <= rel * max(abs(a), abs(b), 1e-12)

# ---- the course's formulas ----
def drift_speed(I, A, n=N_CU): return I / (n * E * A)                    # from I = n e A v
def drift_from_field(Ef, tau): return E * Ef * tau / ME                  # v = e E τ / m
def resistance(rho, L, A): return rho * L / A                            # R = ρ L / A
def at_temp(R0, alpha, T, T0=20.0): return R0 * (1 + alpha * (T - T0))   # R = R₀[1 + α(T − T₀)]
def series(*rs): return sum(rs)                                          # R = R₁ + R₂ + …
def parallel(*rs): return 1 / sum(1 / x for x in rs)                     # 1/R = 1/R₁ + 1/R₂ + …
def terminal(emf, I, r): return emf - I * r                              # V = ε − I r

# ---- reading a typed answer ----
OPS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul, ast.Div: operator.truediv,
       ast.Pow: operator.pow, ast.USub: operator.neg, ast.UAdd: operator.pos}
FUNCS = {"sqrt": math.sqrt}

def calc(text):
    """Plain arithmetic only: numbers (8.5e28 too), + − × ÷, powers, brackets, sqrt and pi."""
    def ev(n):
        if isinstance(n, ast.Expression): return ev(n.body)
        if isinstance(n, ast.Constant) and isinstance(n.value, (int, float)): return n.value
        if isinstance(n, ast.Name) and n.id in ("pi", "π"): return math.pi
        if isinstance(n, ast.BinOp) and type(n.op) in OPS: return OPS[type(n.op)](ev(n.left), ev(n.right))
        if isinstance(n, ast.UnaryOp) and type(n.op) in OPS: return OPS[type(n.op)](ev(n.operand))
        if isinstance(n, ast.Call) and getattr(n.func, "id", "") in FUNCS and len(n.args) == 1:
            return FUNCS[n.func.id](ev(n.args[0]))
        raise ValueError(text)
    return float(ev(ast.parse(text, mode="eval")))

UNIT = (r"\s*(Ω m|ohm m|mm/s|m/s|mAh|Ah|kWh|Wh|°C|degrees?|ohms?|amps?|volts?|watts?|Ω|mA|A|mV|V|kW|W|C|J|s|h|km|mm|cm|m|%|"
        r"times)\s*$")

def read_part(s):
    s = s.strip().replace("−", "-").replace("–", "-").replace("×", "*").replace("÷", "/").replace("^", "**")
    s = s.replace("√", "sqrt")
    t = re.sub(UNIT, "", s).strip()
    try: return calc(t)
    except Exception: return s

def right(got, want, tol):
    return not isinstance(got, str) and abs(got - want) <= max(tol, 1e-7)

def show(v):
    return f"{v:,.10g}".replace(",", " ").replace("-", "−")

# ---- the exercises: the practice set's own (the course's questions and answers) ----
# @EX
EX = [
    {"id": 'P1.1', "tier": 1, "group": 'Electric current and drift',
     "q": 'A steady current carries 18 C of charge past a point in 12 s. What is the current, in amps?',
     "parts": ['current (A)'],
     "answer": [1.5], "tol": [0.005], "function": '/',
     "hint": 'current is the charge passing each second: Q ÷ t',
     "working": 'I = Q/t = 18 / 12 = 1.5 A'},
    {"id": 'P1.2', "tier": 1, "group": 'Electric current and drift',
     "q": 'A phone charger supplies 2 A for 45 minutes. How much charge passes, in coulombs?',
     "parts": ['charge (C)'],
     "answer": [5400.0], "tol": [0.005], "function": '—',
     "hint": 'Q = I × t, with the time in seconds',
     "working": 'Q = I t = 2 × 45 × 60 = 5400 C (that is 1.5 amp-hours)'},
    {"id": 'P1.3', "tier": 1, "group": "Ohm's law and resistance",
     "q": 'A resistor carries 0.25 A when 6 V is across it. What is its resistance, in ohms?',
     "parts": ['resistance (Ω)'],
     "answer": [24.0], "tol": [0.005], "function": '/',
     "hint": 'resistance is the volts needed for each amp',
     "working": 'R = V/I = 6 / 0.25 = 24 Ω'},
    {"id": 'P1.4', "tier": 1, "group": "Ohm's law and resistance",
     "q": "A copper wire is 25 m long and 1.5 mm² across (1.5 × 10⁻⁶ m²). Copper's resistivity is 1.68 × 10⁻⁸ Ω m. What is the wire's resistance, in ohms?",
     "parts": ['resistance (Ω)'],
     "answer": [0.28], "tol": [0.005], "function": '—',
     "hint": 'R = ρ L ÷ A; 1 mm² = 1e-6 m²',
     "working": 'R = ρL/A = 1.68 × 10⁻⁸ × 25 / 1.5 × 10⁻⁶ = 0.2800 Ω'},
    {"id": 'P1.5', "tier": 1, "group": 'Power and combinations',
     "q": 'A lamp carries 2 A on a 12 V supply. What power does it use, in watts?',
     "parts": ['power (W)'],
     "answer": [24.0], "tol": [0.005], "function": '—',
     "hint": 'power = volts × amps',
     "working": 'P = VI = 12 × 2 = 24 W'},
    {"id": 'P1.6', "tier": 1, "group": 'Power and combinations',
     "q": 'Resistors of 3 Ω and 5 Ω are joined in series across 12 V. What current flows, in amps?',
     "parts": ['current (A)'],
     "answer": [1.5], "tol": [0.005], "function": '—',
     "hint": 'the same current through both: add the resistances',
     "working": 'in series the resistances add: 3 + 5 = 8 Ω; I = 12 / 8 = 1.5 A'},
    {"id": 'P1.7', "tier": 1, "group": 'Power and combinations',
     "q": 'Resistors of 6 Ω and 3 Ω are joined in parallel. What is their combined resistance, in ohms?',
     "parts": ['combined resistance (Ω)'],
     "answer": [2.0], "tol": [0.005], "function": '—',
     "hint": 'in parallel the reciprocals add',
     "working": '1/R = 1/6 + 1/3 = 1/2, so R = 2 Ω (less than either)'},
    {"id": 'P1.8', "tier": 1, "group": 'Cells',
     "q": 'A cell of emf 9 V and internal resistance 0.5 Ω drives 2 A. What is the voltage across its terminals, in volts?',
     "parts": ['terminal voltage (V)'],
     "answer": [8.0], "tol": [0.005], "function": '—',
     "hint": 'some of the emf is lost inside the cell: I × r',
     "working": 'V = ε − Ir = 9 − 2 × 0.5 = 8 V'},
    {"id": 'P1.9', "tier": 1, "group": 'Cells',
     "q": 'Four cells, each of emf 1.5 V and internal resistance 0.25 Ω, are joined in series. What are the emf (V) and the internal resistance (Ω) of the battery?',
     "parts": ['emf (V)', 'internal resistance (Ω)'],
     "answer": [6.0, 1.0], "tol": [0.005, 0.005], "function": '—',
     "hint": 'in series, emfs add and internal resistances add',
     "working": 'in series both add: 4 × 1.5 = 6 V and 4 × 0.25 = 1 Ω'},
    {"id": 'P1.10', "tier": 1, "group": "Kirchhoff's rules and the bridge",
     "q": 'At a junction, 5 A flows in along one wire and 2 A flows out along a second. What current flows out along the third wire, in amps?',
     "parts": ['current (A)'],
     "answer": [3.0], "tol": [0.005], "function": '—',
     "hint": 'the junction rule: in = out',
     "working": 'what flows in must flow out: 5 = 2 + I, so I = 3 A'},
    {"id": 'P2.1', "tier": 2, "group": 'Electric current and drift',
     "q": 'A copper cable 2.5 mm² across (2.5 × 10⁻⁶ m²) carries 10 A. Copper has 8.5 × 10²⁸ free electrons per cubic metre, each of charge 1.60 × 10⁻¹⁹ C. What is the drift speed, in mm/s?',
     "parts": ['drift speed (mm/s)'],
     "answer": [0.2941176470588235], "tol": [0.005], "function": '—',
     "hint": 'run I = n e A v backwards: v = I ÷ (n e A)',
     "working": 'v = I/(neA) = 10 / (8.5 × 10²⁸ × 1.60 × 10⁻¹⁹ × 2.5 × 10⁻⁶) = 2.9412e-04 m/s = 0.2941 mm/s'},
    {"id": 'P2.2', "tier": 2, "group": 'Electric current and drift',
     "q": 'How many electrons pass a point each second in a wire carrying 0.8 A? Give the answer in units of 10¹⁸.',
     "parts": ['electrons per second (× 10¹⁸)'],
     "answer": [5.000000000000001], "tol": [0.005], "function": '/',
     "hint": 'charge each second ÷ the charge of one electron',
     "working": 'each second 0.8 C passes; 0.8 / 1.60 × 10⁻¹⁹ = 5 × 10¹⁸ electrons'},
    {"id": 'P2.3', "tier": 2, "group": "Ohm's law and resistance",
     "q": "A copper winding has a resistance of 5.0 Ω at 20 °C. Copper's temperature coefficient is 0.0039 per °C. What is its resistance at 100 °C, in ohms?",
     "parts": ['resistance (Ω)'],
     "answer": [6.5600000000000005], "tol": [0.005], "function": '—',
     "hint": 'the rise is α × the temperature change, as a fraction',
     "working": 'R = R₀[1 + α(T − T₀)] = 5 × (1 + 0.0039 × 80) = 6.5600 Ω'},
    {"id": 'P2.4', "tier": 2, "group": "Ohm's law and resistance",
     "q": "How long a copper wire, 1 mm² across (1.0 × 10⁻⁶ m²), has a resistance of exactly 1 Ω? Copper's resistivity is 1.68 × 10⁻⁸ Ω m. Answer in metres.",
     "parts": ['length (m)'],
     "answer": [59.52380952380952], "tol": [0.005], "function": '—',
     "hint": 'run R = ρL/A backwards',
     "working": 'L = RA/ρ = 1 × 1.0 × 10⁻⁶ / 1.68 × 10⁻⁸ = 59.5238 m'},
    {"id": 'P2.5', "tier": 2, "group": 'Power and combinations',
     "q": "A 2 kW kettle runs on 230 V. What current does it draw (A), and what is its element's resistance (Ω)?",
     "parts": ['current (A)', 'resistance (Ω)'],
     "answer": [8.695652173913043, 26.450000000000003], "tol": [0.005, 0.005], "function": '—',
     "hint": 'P = V I, then R = V ÷ I',
     "working": 'I = P/V = 2000 / 230 = 8.6957 A; R = V/I = 26.4500 Ω'},
    {"id": 'P2.6', "tier": 2, "group": 'Power and combinations',
     "q": 'A 6 Ω and a 3 Ω resistor are in parallel, and that pair is in series with a 2 Ω resistor, across 12 V. What is the total current (A), and the current in the 6 Ω resistor (A)?',
     "parts": ['total current (A)', 'current in the 6 Ω (A)'],
     "answer": [3.0, 1.0], "tol": [0.005, 0.005], "function": '—',
     "hint": 'reduce the pair first, then share the current',
     "working": 'the pair: 2 Ω; with 2 Ω in series: 4 Ω; I = 12 / 4 = 3 A; the pair has 3 × 2 = 6 V across it: 6 / 6 = 1 A in the 6 Ω'},
    {"id": 'P2.7', "tier": 2, "group": 'Cells',
     "q": 'An AA cell of emf 1.5 V and internal resistance 0.2 Ω lights a lamp of resistance 2.8 Ω. What current flows (A), and what is the voltage across the lamp (V)?',
     "parts": ['current (A)', 'voltage across the lamp (V)'],
     "answer": [0.5, 1.4], "tol": [0.005, 0.005], "function": '—',
     "hint": "the cell's own resistance is in series with the lamp",
     "working": 'I = ε/(R + r) = 1.5 / 3.0 = 0.5 A; V = IR = 0.5 × 2.8 = 1.4 V (0.1 V is lost inside the cell)'},
    {"id": 'P2.8', "tier": 2, "group": 'Cells',
     "q": 'Three identical cells, each of emf 1.5 V and internal resistance 0.3 Ω, are joined in parallel and connected to a 1.4 Ω lamp. What current flows in the lamp (A), and in each cell (A)?',
     "parts": ['lamp current (A)', 'current in each cell (A)'],
     "answer": [1.0, 0.3333333333333333], "tol": [0.005, 0.005], "function": '—',
     "hint": 'identical cells side by side: the same emf, the resistance shared',
     "working": 'in parallel: emf 1.5 V, internal resistance 0.3 / 3 = 0.1 Ω; I = 1.5 / 1.5 = 1 A; each cell carries a third: 0.3333 A'},
    {"id": 'P2.9', "tier": 2, "group": "Kirchhoff's rules and the bridge",
     "q": 'Two cells, 12 V and 4 V, are joined in one loop so that they oppose each other, with resistors of 3 Ω and 1 Ω. What current flows, in amps?',
     "parts": ['current (A)'],
     "answer": [2.0], "tol": [0.005], "function": '—',
     "hint": 'the loop rule: the rises and the drops add to zero',
     "working": 'round the loop: 12 − 4 − 3I − I = 0, so I = 8 / 4 = 2 A'},
    {"id": 'P2.10', "tier": 2, "group": "Kirchhoff's rules and the bridge",
     "q": 'A Wheatstone bridge has R₁ = 100 Ω and R₂ = 150 Ω in one arm, and R₃ = 40 Ω with an unknown R₄ in the other. It balances. What is R₄, in ohms?',
     "parts": ['R₄ (Ω)'],
     "answer": [60.0], "tol": [0.005], "function": '—',
     "hint": 'at balance the two arms divide the voltage in the same ratio',
     "working": 'balanced: R₁/R₂ = R₃/R₄, so R₄ = R₃ R₂/R₁ = 40 × 150 / 100 = 60 Ω'},
    {"id": 'P3.1', "tier": 3, "group": 'Electric current and drift',
     "q": 'Electrons drift at 0.2 mm/s along a copper wire 1.5 mm² across (1.5 × 10⁻⁶ m²). Copper has 8.5 × 10²⁸ free electrons per cubic metre, each of charge 1.60 × 10⁻¹⁹ C. What current does the wire carry, in amps?',
     "parts": ['current (A)'],
     "answer": [4.079999999999999], "tol": [0.005], "function": '—',
     "hint": 'I = n e A v, with the speed in m/s',
     "working": 'I = neAv = 8.5 × 10²⁸ × 1.60 × 10⁻¹⁹ × 1.5 × 10⁻⁶ × 2 × 10⁻⁴ = 4.0800 A'},
    {"id": 'P3.2', "tier": 3, "group": "Ohm's law and resistance",
     "q": "A copper winding measures 2.0 Ω at 20 °C. Later, while running, it measures 2.5 Ω. Copper's temperature coefficient is 0.0039 per °C. How hot is the winding, in °C?",
     "parts": ['temperature (°C)'],
     "answer": [84.1025641025641], "tol": [0.005], "function": '—',
     "hint": 'a resistance used as a thermometer: find ΔT, then add it to 20 °C',
     "working": '2.5 / 2.0 = 1 + 0.0039 ΔT, so ΔT = 0.25 / 0.0039 = 64.1026 °C; T = 20 + ΔT = 84.1026 °C'},
    {"id": 'P3.3', "tier": 3, "group": "Ohm's law and resistance",
     "q": 'Copper has resistivity 1.68 × 10⁻⁸ Ω m and 8.5 × 10²⁸ free electrons per cubic metre. Using ρ = m/(ne²τ) with m = 9.11 × 10⁻³¹ kg and e = 1.60 × 10⁻¹⁹ C, find the average time between collisions, τ, in units of 10⁻¹⁴ s.',
     "parts": ['time between collisions (× 10⁻¹⁴ s)'],
     "answer": [2.4920124299719895], "tol": [0.005], "function": '—',
     "hint": 'run ρ = m/(n e² τ) backwards',
     "working": 'τ = m/(ne²ρ) = 9.11 × 10⁻³¹ / (8.5 × 10²⁸ × (1.60 × 10⁻¹⁹)² × 1.68 × 10⁻⁸) = 2.4920e-14 s'},
    {"id": 'P3.4', "tier": 3, "group": 'Power and combinations',
     "q": 'The same 2 kW is sent through a cable of resistance 1 Ω, first at 230 V and then at 2300 V. How much power is lost as heat in the cable each time, in watts?',
     "parts": ['loss at 230 V (W)', 'loss at 2300 V (W)'],
     "answer": [75.61436672967864, 0.7561436672967864], "tol": [0.005, 0.005], "function": '—',
     "hint": 'find the current from P = V I, then the heat from I² R',
     "working": 'at 230 V: I = 2000/230 = 8.70 A, loss I²R = 75.6144 W; at 2300 V: I = 0.870 A, loss 0.7561 W (ten times the voltage, a hundredth of the loss)'},
    {"id": 'P3.5', "tier": 3, "group": 'Power and combinations',
     "q": 'What resistor must be joined in parallel with a 12 Ω resistor to make 4 Ω?',
     "parts": ['resistor (Ω)'],
     "answer": [5.999999999999999], "tol": [0.005], "function": '—',
     "hint": 'run the parallel rule backwards',
     "working": '1/R₂ = 1/4 − 1/12 = 1/6, so R₂ = 6 Ω'},
    {"id": 'P3.6', "tier": 3, "group": 'Cells',
     "q": "A cell's terminal voltage is 1.45 V when it drives 0.2 A, and 1.30 V when it drives 0.8 A. What are its internal resistance (Ω) and its emf (V)?",
     "parts": ['internal resistance (Ω)', 'emf (V)'],
     "answer": [0.2499999999999998, 1.5], "tol": [0.005, 0.005], "function": '—',
     "hint": 'V = ε − Ir is a straight line: its slope is −r',
     "working": 'the voltage falls 0.15 V for 0.6 A more: r = 0.15 / 0.6 = 0.25 Ω; ε = 1.45 + 0.2 × 0.25 = 1.5 V'},
    {"id": 'P3.7', "tier": 3, "group": "Kirchhoff's rules and the bridge",
     "q": "An 11 V cell in line with 2 Ω, and a 9 V cell in line with 2 Ω, both feed the same 4 Ω resistor (each cell's + end joined to the same junction). What current flows in the 11 V cell (A), in the 9 V cell (A), and in the 4 Ω resistor (A)?",
     "parts": ['11 V cell (A)', '9 V cell (A)', '4 Ω resistor (A)'],
     "answer": [1.5, 0.5, 2.0], "tol": [0.005, 0.005, 0.005], "function": '—',
     "hint": 'one junction equation and two loop equations',
     "working": 'junction: I₃ = I₁ + I₂; loops: 11 = 2I₁ + 4I₃ and 9 = 2I₂ + 4I₃; subtract: I₁ − I₂ = 1; add: 20 = 2I₃ + 8I₃, so I₃ = 2 A, I₁ = 1.5 A, I₂ = 0.5 A'},
    {"id": 'P3.8', "tier": 3, "group": 'The e-bike battery',
     "q": "A battery pack is built from cells of 3.6 V, 3.35 Ah and 0.035 Ω each: 13 groups in series, each group 5 cells in parallel. What are the pack's voltage (V), capacity (Ah), internal resistance (Ω), and its terminal voltage when it supplies 15 A (V)?",
     "parts": ['voltage (V)', 'capacity (Ah)', 'internal resistance (Ω)', 'terminal voltage at 15 A (V)'],
     "answer": [46.800000000000004, 16.75, 0.09100000000000001, 45.435], "tol": [0.005, 0.005, 0.005, 0.005], "function": '—',
     "hint": 'series adds volts and resistance; parallel adds capacity and shares resistance',
     "working": 'voltage: 13 × 3.6 = 46.8000 V; capacity: 5 × 3.35 = 16.7500 Ah; each group 0.035/5 = 0.007 Ω, thirteen in series: 0.0910 Ω; V = ε − Ir = 46.80 − 15 × 0.091 = 45.4350 V'},
]
# @END EX
Q = {e["id"]: e for e in EX}
TIER_NAMES = {1: "Warm-up", 2: "Core", 3: "Stretch"}
assert len(EX) == 28 and [sum(e["tier"] == t for e in EX) for t in TIER_NAMES] == [10, 10, 8]

def practice(k, answer=""):
    """Exercise k ("P1.4"): an answer box, a check button and a show-working button."""
    q = Q[k]
    n = len(q["answer"])
    box = Text(value=answer, placeholder="your answer" if n == 1 else f"{n} answers, separated by commas",
               layout=Layout(width="520px"))
    check, work, out = Button(description="check"), Button(description="show working"), Output()
    def on_check(_=None):
        out.clear_output()
        with out:
            got = [read_part(p) for p in re.split(r"[,;]", box.value) if p.strip()]
            if len(got) != n:
                print(f"✗ this one asks for {n} answer(s), in this order: {' · '.join(q['parts'])}. You typed {len(got)}."); return
            ok = [right(g_, w, t) for g_, w, t in zip(got, q["answer"], q["tol"])]
            if all(ok): print("✓ right" if n == 1 else "✓ all right"); return
            for good, part in zip(ok, q["parts"]):
                print(f"  {'✓' if good else '✗'} {part}" + ("" if good else ": not quite"))
            print(textwrap.fill(f"Hint: {q['hint']}", 110))
    def on_work(_=None):
        out.clear_output()
        with out:
            print(textwrap.fill(q["working"], 110))
            print("Answer: " + " · ".join(f"{p}: {show(a)}" for p, a in zip(q["parts"], q["answer"])))
    check.on_click(on_check); work.on_click(on_work)
    print(textwrap.fill(f"{k}. {q['q']}", 110))
    if n > 1: print(textwrap.fill("Answers, in order: " + " · ".join(q["parts"]), 110))
    display(VBox([HBox([box, check, work]), out]))
    if answer: on_check()

def answers(k):
    """The answers the practice set gives for exercise k (for the Python cells below to check against)."""
    return Q[k]["answer"]

def agrees(k, got):
    """True when every value in got is within exercise k's tolerance of its answer."""
    return len(got) == len(answers(k)) and all(right(g_, w, t) for g_, w, t in zip(got, answers(k), Q[k]["tol"]))

print("Ready: e =", E, "C; copper has", N_CU, "free electrons per cubic metre")

## Tier 1 · Warm-up

One idea, one step: current from charge and time, charge from current and time, resistance from a voltage and a
current, the resistance of a wire, a lamp's power, two resistors in series, two in parallel, a cell's terminal
voltage, cells in series, and the junction rule.

In [ ]:
practice("P1.1")

In [ ]:
practice("P1.2")

In [ ]:
practice("P1.3")

In [ ]:
practice("P1.4")

In [ ]:
practice("P1.5")

In [ ]:
practice("P1.6")

In [ ]:
practice("P1.7")

In [ ]:
practice("P1.8")

In [ ]:
practice("P1.9")

In [ ]:
practice("P1.10")

### How you'd do this in Python

Each formula from the course is one short function in the first cell: `drift_speed(I, A)`, `resistance(rho, L, A)`,
`series(R1, R2)`, `parallel(R1, R2)`, `terminal(emf, I, r)` and the rest. Work in amps, volts, ohms, metres and
seconds (1 mm² is `1e-6` m², 45 minutes is `45 * 60` s), then change the unit at the end. Each line below computes one
warm-up answer and checks it against the practice set.

In [ ]:
print(f"P1.4  R = {resistance(RHO_CU, 25, 1.5e-6):.2f} Ω;  P1.7  R = {parallel(6, 3):.2f} Ω")
assert agrees("P1.1", [18 / 12]) and agrees("P1.2", [2 * 45 * 60]) and agrees("P1.3", [6 / 0.25])
assert agrees("P1.4", [resistance(RHO_CU, 25, 1.5e-6)]) and agrees("P1.5", [12 * 2])
assert agrees("P1.6", [12 / series(3, 5)]) and agrees("P1.7", [parallel(6, 3)])
assert agrees("P1.8", [terminal(9, 2, 0.5)]) and agrees("P1.9", [4 * 1.5, 4 * 0.25]) and agrees("P1.10", [5 - 2])
print("Every tier 1 answer, worked out in Python, matches the practice set.")

## Electrons drifting through a wire

Each electron flies freely for a random time, pushed gently by the field, then hits an atom and loses what it
gained. Its path is a zigzag with a tiny lean. Here we follow many electrons through many free flights and measure
the average drift. The course's formula says e E τ / m. A hasty argument would say half of that: the count shows who
is right. Slide the field and the time between collisions: the drift follows both.

In [ ]:
def drift_run(field=0.0252, tau=2.5e-14, electrons=400, flights=200, seed=15):
    """Average drift (m/s) of `electrons` electrons, each through `flights` free flights of random length (mean tau)."""
    rnd = random.Random(seed); a = E * field / ME
    dist = time = 0.0
    for _ in range(electrons * flights):
        t = rnd.expovariate(1 / tau)
        dist += 0.5 * a * t * t; time += t
    return dist / time

def drift_picture(field_mV_per_m=25.2, tau_e14=2.5):
    field, tau = field_mV_per_m / 1000, tau_e14 * 1e-14
    rnd = random.Random(7)
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 4.2), gridspec_kw={"width_ratios": [1.5, 1]})
    fig.subplots_adjust(left=0.05, right=0.98, top=0.82, bottom=0.14, wspace=0.25)
    lean = 0.18 * (field / 0.0252) * (tau / 2.5e-14)               # the lean is exaggerated so that you can see it
    for k in range(10):
        x, y = rnd.uniform(-1, 1), rnd.uniform(-1, 1); xs, ys = [x], [y]
        for _ in range(40):
            th = rnd.uniform(0, 2 * math.pi); step = rnd.expovariate(1.0)
            x += step * math.cos(th) + lean * step * step; y += step * math.sin(th); xs.append(x); ys.append(y)
        a1.plot(xs, ys, color=CHALK if k else GOLD, lw=0.9 if k else 1.6, alpha=0.7 if k else 1)
    a1.set_xticks([]); a1.set_yticks([]); a1.grid(False); a1.set_title("ten electrons (the lean is exaggerated)", color=DIM, fontsize=12)
    sim, formula = drift_run(field, tau), drift_from_field(field, tau)
    a2.bar(["counted", "e E τ / m", "half of it"], [sim * 1e3, formula * 1e3, formula * 0.5e3], color=[GOLD, CHALK, DIM])
    a2.set_ylabel("drift (mm/s)")
    fig.suptitle(f"drift counted over 80,000 free flights: {sim * 1e3:.4f} mm/s;  e E τ / m = {formula * 1e3:.4f} mm/s",
                 x=0.05, ha="left", color=GOLD, fontsize=13)
    footnote(fig); plt.show()
    return sim, formula

sim_d, formula_d = drift_picture()
assert close(sim_d, formula_d, 0.05) and abs(sim_d / (0.5 * formula_d) - 1) > 0.5
assert close(drift_speed(1.5, 1e-6), formula_d, 0.01)             # the same wire by I = n e A v: 1.5 A in 1 mm² of copper

In [ ]:
interact(drift_picture, field_mV_per_m=FloatSlider(value=25.2, min=5.0, max=100.0, step=5.0, description="E (mV/m)"),
         tau_e14=FloatSlider(value=2.5, min=0.5, max=5.0, step=0.5, description="τ (10⁻¹⁴ s)"));

## Tier 2 · Core

Several steps each: a drift speed, electrons per second, a warm winding, a length of wire, a kettle, a mixed network,
a cell lighting a lamp, cells in parallel, two cells in one loop, and a balanced bridge.

In [ ]:
practice("P2.1")

In [ ]:
practice("P2.2")

In [ ]:
practice("P2.3")

In [ ]:
practice("P2.4")

In [ ]:
practice("P2.5")

In [ ]:
practice("P2.6")

In [ ]:
practice("P2.7")

In [ ]:
practice("P2.8")

In [ ]:
practice("P2.9")

In [ ]:
practice("P2.10")

### How you'd do this in Python

Reduce a network one pair at a time: `parallel(6, 3)` first, then `series(...)`. A cell's own resistance is simply
one more resistor in series with whatever it feeds.

In [ ]:
assert agrees("P2.1", [drift_speed(10, 2.5e-6) * 1000]) and agrees("P2.2", [0.8 / E / 1e18])
assert agrees("P2.3", [at_temp(5.0, ALPHA_CU, 100)]) and agrees("P2.4", [1 * 1.0e-6 / RHO_CU])
I_kettle = 2000 / 230
assert agrees("P2.5", [I_kettle, 230 / I_kettle])
R_net = series(parallel(6, 3), 2); I_net = 12 / R_net
print(f"P2.6  total {I_net:.2f} A; the pair has {I_net * parallel(6, 3):.2f} V across it")
assert agrees("P2.6", [I_net, I_net * parallel(6, 3) / 6])
I_cell = 1.5 / (2.8 + 0.2)
assert agrees("P2.7", [I_cell, I_cell * 2.8]) and agrees("P2.8", [1.5 / (1.4 + 0.3 / 3), 1.5 / (1.4 + 0.3 / 3) / 3])
assert agrees("P2.9", [(12 - 4) / (3 + 1)]) and agrees("P2.10", [40 * 150 / 100])
print("Every tier 2 answer, worked out in Python, matches the practice set.")

## A cell under load

A real cell behaves like a perfect cell with a small resistor inside it. The more current you draw, the more voltage
is lost inside: the terminal voltage falls along a straight line, V = ε − I r. Where the line starts is the EMF; its
slope is the internal resistance. The second picture shows the power the cell delivers to a load: it peaks when the
load's resistance equals the cell's own. Slide the EMF and the internal resistance.

In [ ]:
def cell_picture(emf=1.5, r=0.2, R_load=2.8):
    I_short = emf / r
    I = np.linspace(0, I_short, 100)
    I_now = emf / (R_load + r)
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 4.0))
    fig.subplots_adjust(left=0.08, right=0.98, top=0.80, bottom=0.17, wspace=0.28)
    a1.plot(I, emf - I * r, color=CHALK); a1.plot([I_now], [terminal(emf, I_now, r)], "o", color=GOLD, ms=10)
    a1.set_xlabel("current (A)"); a1.set_ylabel("terminal voltage (V)")
    a1.set_title(f"starts at ε = {emf:g} V, slope −r = −{r:g} Ω", color=DIM, fontsize=12)
    R = np.linspace(0.01, max(3.0, 6 * r), 300)
    a2.plot(R, (emf / (R + r)) ** 2 * R, color=VIOLET); a2.axvline(r, color=GOLD, lw=1.2, ls="--")
    a2.set_xlabel("load resistance (Ω)"); a2.set_ylabel("power in the load (W)")
    a2.set_title("the load gets most power when R = r", color=DIM, fontsize=12)
    fig.suptitle(f"with a {R_load:g} Ω load: {I_now:.3f} A, {terminal(emf, I_now, r):.3f} V at the terminals, "
                 f"{I_now * r:.3f} V lost inside", x=0.08, ha="left", color=GOLD, fontsize=13)
    footnote(fig, "computed"); plt.show()
    best = max(R, key=lambda x: (emf / (x + r)) ** 2 * x)
    return I_now, terminal(emf, I_now, r), best

I_c, V_c, R_best = cell_picture()
assert close(I_c, 0.5, 1e-9) and close(V_c, 1.4, 1e-9) and abs(R_best - 0.2) < 0.02

In [ ]:
interact(cell_picture, emf=FloatSlider(value=1.5, min=1.0, max=12.0, step=0.5, description="ε (V)"),
         r=FloatSlider(value=0.2, min=0.05, max=1.0, step=0.05, description="r (Ω)"),
         R_load=FloatSlider(value=2.8, min=0.2, max=10.0, step=0.2, description="load (Ω)"));

## Tier 3 · Stretch

Run a formula backwards, combine two ideas, spot the trap: a current from a drift speed, a winding used as a
thermometer, the time between collisions, the heat lost in a cable, a parallel resistor, a cell from two readings, a
two-cell network, and a bigger battery pack.

In [ ]:
practice("P3.1")

In [ ]:
practice("P3.2")

In [ ]:
practice("P3.3")

In [ ]:
practice("P3.4")

In [ ]:
practice("P3.5")

In [ ]:
practice("P3.6")

In [ ]:
practice("P3.7")

In [ ]:
practice("P3.8")

### How you'd do this in Python

For the two-cell network, write the junction rule and the two loop rules, then solve. Below, the same network is
also handed to `settle`, which uses no loop equations at all: the two answers agree.

In [ ]:
assert agrees("P3.1", [N_CU * E * 1.5e-6 * 2e-4]) and agrees("P3.2", [20 + (2.5 / 2.0 - 1) / ALPHA_CU])
assert agrees("P3.3", [ME / (N_CU * E ** 2 * RHO_CU) / 1e-14])
assert agrees("P3.4", [(2000 / 230) ** 2 * 1, (2000 / 2300) ** 2 * 1]) and agrees("P3.5", [1 / (1 / 4 - 1 / 12)])
r_cell = (1.45 - 1.30) / (0.8 - 0.2)
assert agrees("P3.6", [r_cell, 1.45 + 0.2 * r_cell])
# P3.7 by hand: I3 = I1 + I2; 11 = 2 I1 + 4 I3; 9 = 2 I2 + 4 I3. Add the loops: 20 = 2 I3 + 8 I3.
I3 = 20 / 10; I1 = (11 - 4 * I3) / 2; I2 = (9 - 4 * I3) / 2
V_, I_ = settle(["g", "t"], [("g", "t", 2, 11), ("g", "t", 2, 9), ("t", "g", 4, 0)], {"g": 0.0})
print(f"P3.7  by hand: {I1:.2f}, {I2:.2f}, {I3:.2f} A;  settled: {I_[0]:.2f}, {I_[1]:.2f}, {I_[2]:.2f} A")
assert agrees("P3.7", [I1, I2, I3]) and agrees("P3.7", I_)
assert agrees("P3.8", [13 * 3.6, 5 * 3.35, 13 * 0.035 / 5, 13 * 3.6 - 15 * 13 * 0.035 / 5])
print("Every tier 3 answer, worked out in Python, matches the practice set.")

## Kirchhoff's rules: a circuit that settles

Two cells, each with a resistor, feed one shared resistor: a circuit that series and parallel can't untangle. By
hand, the junction rule and two loop rules give the three currents. The solver takes another road: it nudges the
junction's potential until what flows in equals what flows out. Slide the cells and the shared resistor. A negative
current means that cell is being charged.

In [ ]:
def two_cells_by_hand(e1, R1, e2, R2, R3):
    det = (R1 + R3) * (R2 + R3) - R3 * R3
    i1 = (e1 * (R2 + R3) - e2 * R3) / det; i2 = (e2 * (R1 + R3) - e1 * R3) / det
    return i1, i2, i1 + i2

def network_picture(e1=12.0, e2=6.0, R3=4.0, R1=2.0, R2=2.0):
    V, I = settle(["g", "t"], [("g", "t", R1, e1), ("g", "t", R2, e2), ("t", "g", R3, 0)], {"g": 0.0})
    hand = two_cells_by_hand(e1, R1, e2, R2, R3)
    path = [0.0]
    for _ in range(25):                                            # the same settling, slowed down so you can watch it
        target = (e1 / R1 + e2 / R2) / (1 / R1 + 1 / R2 + 1 / R3); path.append(path[-1] + 0.3 * (target - path[-1]))
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 4.0))
    fig.subplots_adjust(left=0.08, right=0.98, top=0.80, bottom=0.17, wspace=0.28)
    a1.plot(path, color=GOLD, marker="o", ms=4); a1.axhline(V["t"], color=DIM, lw=1, ls="--")
    a1.set_xlabel("sweep"); a1.set_ylabel("the junction's potential (V)"); a1.set_title("settling", color=DIM, fontsize=12)
    a2.bar(["I₁", "I₂", "I₃"], I, color=[SKY if x >= 0 else CORAL for x in I]); a2.axhline(0, color=DIM, lw=1)
    a2.set_ylabel("current (A)"); a2.set_title("settled currents", color=DIM, fontsize=12)
    fig.suptitle(f"I₁ = {I[0]:.3f} A, I₂ = {I[1]:.3f} A, I₃ = {I[2]:.3f} A   (by hand: {hand[0]:.3f}, {hand[1]:.3f}, "
                 f"{hand[2]:.3f})", x=0.08, ha="left", color=GOLD, fontsize=13)
    footnote(fig, "settled junction by junction · checked against Kirchhoff's rules"); plt.show()
    return I, hand, e1 * I[0] + e2 * I[1] - (I[0] ** 2 * R1 + I[1] ** 2 * R2 + I[2] ** 2 * R3)

I_s, I_h, leftover = network_picture()
assert all(close(a, b, 1e-9) for a, b in zip(I_s, I_h)) and [round(x, 6) for x in I_s] == [2.4, -0.6, 1.8]
assert abs(I_s[0] + I_s[1] - I_s[2]) < 1e-9 and abs(leftover) < 1e-9      # the junction rule, and every joule accounted for

In [ ]:
interact(network_picture, e1=FloatSlider(value=12.0, min=1.0, max=24.0, step=1.0, description="ε₁ (V)"),
         e2=FloatSlider(value=6.0, min=1.0, max=24.0, step=1.0, description="ε₂ (V)"),
         R3=FloatSlider(value=4.0, min=0.5, max=20.0, step=0.5, description="R₃ (Ω)"),
         R1=FloatSlider(value=2.0, min=0.5, max=10.0, step=0.5, description="R₁ (Ω)"),
         R2=FloatSlider(value=2.0, min=0.5, max=10.0, step=0.5, description="R₂ (Ω)"));

## The Wheatstone bridge

Four resistors in a diamond, a battery across the top and bottom, and a galvanometer between the two side corners.
Slide the fourth resistor: the galvanometer's current falls through zero exactly where R₁/R₂ = R₃/R₄. Change the
battery: the balance point doesn't move.

In [ ]:
def bridge(R1, R2, R3, R4, G=10.0, V=6.0):
    """The settled bridge: the galvanometer's current, from the left corner to the right (A)."""
    Vn, I = settle(["A", "B", "C", "D"], [("A", "B", R1, 0), ("B", "C", R2, 0), ("A", "D", R3, 0), ("D", "C", R4, 0),
                                          ("B", "D", G, 0)], {"A": V, "C": 0.0})
    return I[4]

def bridge_picture(R4=40.0, battery=6.0, R1=10.0, R2=20.0, R3=15.0):
    xs = np.linspace(5, 80, 76)
    ig = [bridge(R1, R2, R3, x, V=battery) * 1000 for x in xs]
    now = bridge(R1, R2, R3, R4, V=battery) * 1000
    balance = R3 * R2 / R1
    fig, ax = plt.subplots(figsize=(9.5, 4.2))
    fig.subplots_adjust(left=0.10, right=0.98, top=0.84, bottom=0.17)
    ax.plot(xs, ig, color=CHALK); ax.axhline(0, color=DIM, lw=1); ax.axvline(balance, color=GOLD, lw=1.2, ls="--")
    ax.plot([R4], [now], "o", color=GOLD, ms=10)
    ax.set_xlabel("R₄ (Ω)"); ax.set_ylabel("galvanometer current (mA)")
    fig.suptitle(f"R₄ = {R4:g} Ω: {now:+.2f} mA;  balance at R₃R₂/R₁ = {balance:g} Ω", x=0.10, ha="left", color=GOLD)
    footnote(fig, "settled junction by junction"); plt.show()
    return now, bridge(R1, R2, R3, balance, V=battery)

ig_40, ig_bal = bridge_picture()
assert round(ig_40, 1) == -13.2 and abs(ig_bal) < 1e-9

In [ ]:
interact(bridge_picture, R4=FloatSlider(value=40.0, min=5.0, max=80.0, step=1.0, description="R₄ (Ω)"),
         battery=FloatSlider(value=6.0, min=1.5, max=12.0, step=1.5, description="battery (V)"),
         R1=FloatSlider(value=10.0, min=5.0, max=50.0, step=5.0, description="R₁ (Ω)"),
         R2=FloatSlider(value=20.0, min=5.0, max=50.0, step=5.0, description="R₂ (Ω)"),
         R3=FloatSlider(value=15.0, min=5.0, max=50.0, step=5.0, description="R₃ (Ω)"));

## The e-bike battery: rebuild the pack

The course's opening question. Each cell gives 3.6 V, holds 3.35 Ah and has about 0.035 Ω of internal resistance.
Cells in series add their volts; cells in parallel add their amp-hours and share the current. Slide the layout and
the current the motor draws: the pack's voltage, the voltage lost inside it, the heat, the running time and the
distance all follow. (The distance assumes a steady 25 km/h with the motor doing the work; real riders pedal too.)

In [ ]:
def pack(in_series=10, in_parallel=4, I=7.0, cell_V=3.6, cell_Ah=3.35, cell_r=0.035, kmh=25.0):
    emf, Ah, r = in_series * cell_V, in_parallel * cell_Ah, in_series * cell_r / in_parallel
    V = terminal(emf, I, r)
    return {"emf": emf, "Ah": Ah, "r": r, "V": V, "each": I / in_parallel, "heat": I * I * r, "hours": Ah / I,
            "km": Ah / I * kmh, "Wh": emf * Ah}

def pack_picture(in_series=10, in_parallel=4, I=7.0):
    p = pack(int(in_series), int(in_parallel), I)
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 4.1), gridspec_kw={"width_ratios": [1.3, 1]})
    fig.subplots_adjust(left=0.03, right=0.98, top=0.78, bottom=0.22, wspace=0.42)
    for i in range(int(in_series)):
        for j in range(int(in_parallel)):
            a1.add_patch(plt.Rectangle((i * 1.2, j * 0.5), 1.0, 0.36, facecolor=GHOST, edgecolor=GOLD if (i, j) == (0, 0) else CHALK, lw=1.2))
    a1.set_xlim(-0.3, int(in_series) * 1.2 + 0.1); a1.set_ylim(-0.3, max(2.5, int(in_parallel) * 0.5 + 0.2))
    a1.set_xticks([]); a1.set_yticks([]); a1.grid(False)
    a1.set_title(f"{int(in_series)} in series × {int(in_parallel)} in parallel = {int(in_series) * int(in_parallel)} cells",
                 color=DIM, fontsize=12)
    a2.barh(["lost inside", "at the motor"], [p["emf"] - p["V"], p["V"]], color=[VIOLET, CHALK]); a2.set_xlabel("volts")
    fig.suptitle(f"{p['emf']:.1f} V · {p['Ah']:.2f} Ah · {p['Wh']:.0f} Wh · r = {p['r']:.4f} Ω · at {I:g} A: "
                 f"{p['V']:.2f} V, {p['each']:.2f} A a cell, {p['heat']:.1f} W of heat, {p['hours']:.2f} h, "
                 f"{p['km']:.0f} km", x=0.05, ha="left", color=GOLD, fontsize=12)
    footnote(fig, "computed · cell: 3.6 V, 3.35 Ah, 0.035 Ω"); plt.show()
    return p

p0 = pack_picture()
assert round(p0["emf"], 6) == 36 and round(p0["Ah"], 6) == 13.4 and round(p0["V"], 1) == 35.4 and p0["each"] == 1.75
assert round(p0["hours"], 1) == 1.9 and round(p0["km"]) == 48 and round(p0["Wh"]) == 482
# the same pack as a network of forty real cells, settled: the same terminal voltage
nodes = [f"n{k}" for k in range(11)]
R_load = p0["V"] / 7.0
branches = [(f"n{k}", f"n{k + 1}", 0.035, 3.6) for k in range(10) for _ in range(4)] + [("n10", "n0", R_load, 0)]
V_pack, I_pack = settle(nodes, branches, {"n0": 0.0}, sweeps=200000, tol=1e-12)
assert close(V_pack["n10"], p0["V"], 1e-6) and close(I_pack[0], 1.75, 1e-6)

In [ ]:
interact(pack_picture, in_series=IntSlider(value=10, min=1, max=14, step=1, description="in series"),
         in_parallel=IntSlider(value=4, min=1, max=6, step=1, description="in parallel"),
         I=FloatSlider(value=7.0, min=1.0, max=20.0, step=0.5, description="current (A)"));

## Sources
Every exercise and set-up here is our own. The constants are the course's rounded values (the electron's charge and
mass from CODATA 2022, physics.nist.gov/cuu/Constants; copper's and nichrome's resistivities and temperature
coefficients from OpenStax University Physics volume 2, Table 9.1; copper's 8.5 × 10²⁸ free electrons per cubic metre
from its density and atomic mass, one free electron per atom). The e-bike pack is a 36 V, 13.4 Ah pack of forty cells
(the Bosch PowerPack 500's rating); the cell is a typical 18650 cell (the Samsung INR18650-35E specification: 3.6 V,
at least 3,350 mAh, 35 mΩ measured with AC, 8 A continuous). The 7 A load and the 25 km/h are round figures near the
European limits for a pedal-assist bicycle (250 W, 25 km/h).

In [ ]:
# When this notebook runs beside the course's source files, this cell checks it still matches them; elsewhere it does
# nothing.
import importlib.util, pathlib, sys
_src = pathlib.Path("../../../episodes/ph15_current/practice.py")
if _src.exists():
    sys.path.insert(0, str(_src.resolve().parents[2]))
    _spec = importlib.util.spec_from_file_location("ph15_practice", _src)
    _P = importlib.util.module_from_spec(_spec); _spec.loader.exec_module(_P)
    def _flat(a): return [y for x in a for y in _flat(x)] if isinstance(a, (tuple, list)) else [a]
    for _e in _P.EX:
        _q = Q[_e["id"]]
        assert (_q["q"], _q["working"], _q["function"], _q["tier"], _q["parts"], _q["hint"]) == (
            _e["q"], _e["working"], _e["function"], _e["tier"], _e["parts"], _e["hint"]), _e["id"]
        assert len(_q["answer"]) == len(_flat(_e["answer"])) and all(
            abs(a - float(b)) < 1e-9 * max(1, abs(b)) for a, b in zip(_q["answer"], _flat(_e["answer"]))), _e["id"]
    assert len(_P.EX) == len(EX)
    assert (E, ME, N_CU, RHO_CU, ALPHA_CU, RHO_NICHROME) == (_P.E, _P.ME, _P.N_CU, _P.RHO["copper"], _P.ALPHA["copper"],
                                                             _P.RHO["nichrome"])
    assert close(_P.EBIKE["V_term"], p0["V"], 1e-12) and close(_P.NET["I1"], I_s[0], 1e-9) and close(_P.WB["unbal"][2] * 1000, ig_40, 1e-9)
    print("matches the course's practice set")